# Combining Two Ukrainian Datasets

**Note:** We combine two Ukrainian-language datasets.  
The first consists of approximately **1,000 Ukrainian comments** collected from X (formerly Twitter) using **Apify**.  
The second dataset, sourced from **Hugging Face**, contains **5,000 Twitter comments** (2,500 toxic and 2,500 non-toxic) and was originally used in a **study on Ukrainian binary toxicity classification** ([link](https://huggingface.co/datasets/ukr-detect/ukr-toxicity-dataset)). This dataset was introduced in the paper [*“Toxicity Classification in Ukrainian”*](https://aclanthology.org/2024.woah-1.19.pdf).

## Importing Dependencies

In [1]:
import pandas as pd
from datasets import load_dataset

from toxicity_detector.config.paths import UKR_RAW
from toxicity_detector.config.labels import LABELS_EN

c:\Users\Amina\Desktop\Thesis project\PROJECT\my_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading Data

In [2]:
df = pd.read_csv(UKR_RAW["comments"])
hf_dataset = load_dataset("ukr-detect/ukr-toxicity-dataset")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1039 entries, 0 to 1038
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   comment        1039 non-null   object
 1   toxic          1039 non-null   int64 
 2   severe_toxic   1039 non-null   int64 
 3   obscene        1039 non-null   int64 
 4   threat         1039 non-null   int64 
 5   insult         1039 non-null   int64 
 6   identity_hate  1039 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 56.9+ KB


In [4]:
hf_df = hf_dataset["train"].to_pandas()
hf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5000 non-null   object
 1   toxic   5000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 78.2+ KB


In [5]:
hf_df = hf_df.rename(columns={"text": "comment"})

In [6]:
for col in LABELS_EN[1:]:
    hf_df[col] = 0
    
hf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   comment        5000 non-null   object
 1   toxic          5000 non-null   int64 
 2   severe_toxic   5000 non-null   int64 
 3   obscene        5000 non-null   int64 
 4   threat         5000 non-null   int64 
 5   insult         5000 non-null   int64 
 6   identity_hate  5000 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 273.6+ KB


In [7]:
hf_df[["comment"]].sample(10)

,comment
2709,"Бла, чую день буде хардовий шо піздєц)0)"
806,"Їм це картинка, багато лайків, які напевно є ї..."
3999,"Боже-Боже, дві довбойобки в стрічці, чесне слово"
1789,"площа франка, напрімєр) це рівно по серединці ..."
4451,"Що там, блять, за скотина собі самооцінку піді..."
3516,це кому там треба піздюлів вламати?
216,"RT Папая, кая і маракуя #np the Вйо - Їдемо в..."
1271,Звемо її Баблотрекер ))) Сьогодні перший тесто...
1014,"Кіріл чи Кирило, вам не пофіг?"
4763,Метелики у животі вічно працюють проти мене Ко...


## Combining Datasets

In [14]:
def combine_datasets(df1, df2, output_path):    
    df = pd.concat([df1, df2], ignore_index=True)
    
    df.to_csv(output_path, index=True, index_label="id")

In [15]:
# Combining datasets with only comments
combine_datasets(df[["comment"]], hf_df[["comment"]], UKR_RAW["combined"])

In [16]:
# Combining datasets with only labels
combine_datasets(df[LABELS_EN], hf_df[LABELS_EN], UKR_RAW["labels"])